# NumPy Intermediate — Notebook 2 of 3

## Table of Contents
1. Boolean Indexing
2. Fancy Indexing
3. np.where()
4. np.nonzero() and np.argwhere()
5. Broadcasting (Deep Dive)
6. clip(), cumsum(), cumprod(), diff()
7. NaN and Inf Tools
8. Math Functions
9. Linear Algebra — Matrix Multiply
10. Linear Algebra — Other Operations
11. SVD — Singular Value Decomposition
12. Statistics
13. Sorting and Searching
14. Stacking and Splitting
15. np.pad()
16. np.einsum() — Einstein Summation
17. np.vectorize()
18. Random Module (Deep Dive)
19. Copy vs View
20. Cheat Sheet
21. Exercises


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)


## Section 1: Boolean Indexing
Filter arrays based on conditions — used in data preprocessing


Create boolean mask from condition — `arr > 5`


In [ ]:
arr = np.array([1, 4, 7, 2, 9, 8])
mask = arr > 5
print('Mask:', mask)


Use mask to filter array elements


In [ ]:
filtered_arr = arr[mask]
print('Filtered:', filtered_arr)


Boolean indexing on 2D array — filter rows


In [ ]:
arr_2d = np.array([[1, 2], [5, 6], [8, 9]])
row_mask = arr_2d[:, 0] > 3
filtered_rows = arr_2d[row_mask]
print('Filtered Rows:\n', filtered_rows)


Real use — filter out outliers (scores > 100 or < 0) from a dataset


In [ ]:
scores = np.array([-5, 85, 92, 110, 45, 105])
valid_mask = (scores >= 0) & (scores <= 100)
clean_scores = scores[valid_mask]
print('Clean Scores:', clean_scores)


Combine conditions with & (and) and | (or)


In [ ]:
arr = np.arange(10)
combined_mask = (arr > 2) & (arr < 7)
print('Values between 2 and 7:', arr[combined_mask])


## Section 2: Fancy Indexing
Index with an array of indices


1D fancy indexing — `arr[[0, 3, 5]]`


In [ ]:
arr = np.arange(10, 20)
selected = arr[[0, 3, 5]]
print('Selected elements:', selected)


2D fancy indexing — select specific rows


In [ ]:
arr_2d = np.arange(1, 10).reshape(3, 3)
rows = arr_2d[[0, 2]]
print('Selected rows:\n', rows)


Select specific rows AND specific columns


In [ ]:
arr_2d = np.arange(1, 10).reshape(3, 3)
# Select element at (0,1) and (2,2)
elements = arr_2d[[0, 2], [1, 2]]
print('Selected elements:', elements)


Real use — select a random subset of training samples


In [ ]:
dataset = np.arange(100).reshape(10, 10)
indices = np.random.choice(dataset.shape[0], size=3, replace=False)
random_batch = dataset[indices]
print('Random batch rows:\n', random_batch)


## Section 3: np.where()
np.where — conditional element selection


`np.where(condition, x, y)` — return x where True, y where False


In [ ]:
arr = np.array([1, 5, 2, 8, 3])
result = np.where(arr > 4, 'High', 'Low')
print(result)


`np.where(condition)` — get indices of True elements


In [ ]:
arr = np.array([1, 5, 2, 8, 3])
indices = np.where(arr > 4)
print('Indices:', indices)


Real ML use — np.where to apply ReLU activation (replace negatives with 0)


In [ ]:
layer_outputs = np.array([-1.5, 2.0, -0.5, 3.1, 0.0])
relu_outputs = np.where(layer_outputs > 0, layer_outputs, 0)
print('ReLU outputs:', relu_outputs)


Real use — label binarization: score>60 → 1 else 0


In [ ]:
scores = np.array([45, 80, 65, 30, 95])
labels = np.where(scores > 60, 1, 0)
print('Binary labels:', labels)


## Section 4: np.nonzero() and np.argwhere()


`np.nonzero(arr)` — indices of non-zero elements


In [ ]:
arr = np.array([0, 2, 0, 4, 0])
print('Non-zero indices:', np.nonzero(arr))


`np.argwhere(condition)` — 2D array of indices satisfying condition


In [ ]:
arr_2d = np.array([[1, 0], [0, 5]])
indices = np.argwhere(arr_2d > 0)
print('Indices > 0:\n', indices)


Real use — find positions of missing data (NaN values)


In [ ]:
data = np.array([1.0, np.nan, 3.5, np.nan, 5.0])
nan_positions = np.argwhere(np.isnan(data))
print('NaN positions:\n', nan_positions)


## Section 5: Broadcasting (Deep Dive)
Broadcasting rules with ASCII art visual:
```
Rule 1: If arrays have different ndim, prepend 1s to smaller shape
Rule 2: Dimensions of size 1 are stretched to match the other
Rule 3: Shapes must be compatible after rules 1 & 2
```


Scalar broadcast — `arr + 5`


In [ ]:
arr = np.array([1, 2, 3])
print('arr + 5:', arr + 5)


1D broadcast over 2D — add a vector to each row


In [ ]:
arr_2d = np.ones((3, 4))
vector = np.array([1, 2, 3, 4])
print('2D + 1D:\n', arr_2d + vector)


Column vector broadcast — subtract column mean (normalize each feature)


In [ ]:
arr_2d = np.ones((3, 4))
col_vector = np.array([[1], [2], [3]])
print('2D + Col Vector:\n', arr_2d + col_vector)


3D broadcasting example — batch of matrices


In [ ]:
batch = np.ones((2, 3, 4))
vector = np.array([1, 2, 3, 4])
print('Batch + Vector shape:', (batch + vector).shape)


Common mistake — shape mismatch error with fix


In [ ]:
a = np.ones((3, 4))
b = np.ones(3)
try:
    print(a + b)
except ValueError as e:
    print('Error:', e)
# Fix by reshaping b to a column vector
print('\nFixed shape:', (a + b.reshape(-1, 1)).shape)


Real ML use — subtract mean of each feature column (vectorized)


In [ ]:
X = np.array([[1, 2], [3, 4], [5, 6]])
feature_means = X.mean(axis=0)
X_centered = X - feature_means
print('Centered X:\n', X_centered)


Real ML use — batch normalization step (subtract mean, divide by std)


In [ ]:
feature_stds = X.std(axis=0)
X_normalized = (X - feature_means) / feature_stds
print('Normalized X:\n', X_normalized)


## Section 6: clip(), cumsum(), cumprod(), diff()


`np.clip(arr, min, max)` — clamp values to range


In [ ]:
arr = np.array([-5, 0, 5, 10, 15])
print('Clipped:', np.clip(arr, 0, 10))


ML use of clip — clip probabilities to [1e-7, 1-1e-7] before log() to avoid log(0)


In [ ]:
probs = np.array([0.0, 0.5, 1.0])
clipped_probs = np.clip(probs, 1e-7, 1 - 1e-7)
print('Safe log:', np.log(clipped_probs))


ML use of clip — gradient clipping to prevent exploding gradients


In [ ]:
gradients = np.array([-150.0, -1.0, 0.5, 200.0])
clipped_grads = np.clip(gradients, -5.0, 5.0)
print('Clipped gradients:', clipped_grads)


`np.cumsum()` — running total along axis


In [ ]:
arr = np.array([1, 2, 3, 4])
print('Cumulative sum:', np.cumsum(arr))


`np.cumprod()` — running product


In [ ]:
arr = np.array([1, 2, 3, 4])
print('Cumulative product:', np.cumprod(arr))


`np.diff()` — differences between consecutive elements


In [ ]:
arr = np.array([1, 3, 6, 10])
print('Differences:', np.diff(arr))


Real use — compute daily returns from stock prices using np.diff


In [ ]:
prices = np.array([100, 102, 101, 105, 110])
returns = np.diff(prices) / prices[:-1]
print('Daily returns:', returns)


## Section 7: NaN and Inf Tools


`np.isnan(arr)` — boolean mask of NaN positions


In [ ]:
arr = np.array([1.0, np.nan, 3.0])
print('Is NaN:', np.isnan(arr))


`np.isinf(arr)` — boolean mask of Inf positions


In [ ]:
arr = np.array([1.0, np.inf, -np.inf])
print('Is Inf:', np.isinf(arr))


`np.isfinite(arr)` — boolean mask of finite positions


In [ ]:
arr = np.array([1.0, np.nan, np.inf])
print('Is finite:', np.isfinite(arr))


`np.nan_to_num(arr, nan=0, posinf=1e6, neginf=-1e6)`


In [ ]:
arr = np.array([1.0, np.nan, np.inf, -np.inf])
print('Cleaned:', np.nan_to_num(arr, nan=0.0, posinf=1e6, neginf=-1e6))


Real use — clean a dataset: remove rows with NaN using boolean indexing


In [ ]:
data = np.array([[1, 2], [3, np.nan], [5, 6]])
mask = ~np.isnan(data).any(axis=1)
clean_data = data[mask]
print('Clean data:\n', clean_data)


`np.allclose(a, b)` — compare arrays with tolerance (for testing)


In [ ]:
a = np.array([0.1 + 0.2])
b = np.array([0.3])
print('Exact match:', a == b)
print('All close:', np.allclose(a, b))


`np.isclose(a, b)` — element-wise approximate equality


In [ ]:
a = np.array([1e-9, 1.0, 2.0])
b = np.array([0.0, 1.0 + 1e-10, 2.1])
print('Is close:', np.isclose(a, b, atol=1e-8))


`np.count_nonzero(arr)` — count non-zero elements (sparsity measure)


In [ ]:
arr = np.array([0, 1, 0, 0, 5, 0])
print('Non-zero count:', np.count_nonzero(arr))
print('Sparsity:', 1 - (np.count_nonzero(arr) / arr.size))


## Section 8: Math Functions
Overview of universal functions (ufuncs)


Trigonometric — `np.sin`, `np.cos`, `np.tan` on array


In [ ]:
angles = np.array([0, np.pi/2, np.pi])
print('Sine:', np.sin(angles))


Inverse trig — `np.arcsin`, `np.arccos`, `np.arctan`


In [ ]:
vals = np.array([0, 1])
print('Arcsin:', np.arcsin(vals))


`np.exp(arr)` — e^x, used in softmax, sigmoid


In [ ]:
arr = np.array([0, 1, 2])
print('e^x:', np.exp(arr))


`np.log(arr)` — natural log, used in cross-entropy loss


In [ ]:
arr = np.array([1, np.e, np.e**2])
print('ln(x):', np.log(arr))


`np.log2`, `np.log10`


In [ ]:
arr = np.array([1, 10, 100])
print('log10:', np.log10(arr))


`np.sqrt(arr)` — square root


In [ ]:
arr = np.array([1, 4, 9])
print('sqrt:', np.sqrt(arr))


`np.power(arr, n)` — element-wise power


In [ ]:
arr = np.array([1, 2, 3])
print('power(x, 2):', np.power(arr, 2))


`np.abs(arr)` — absolute value (used in L1 loss)


In [ ]:
arr = np.array([-1, -5, 3])
print('abs:', np.abs(arr))


`np.floor`, `np.ceil`, `np.round` — rounding


In [ ]:
arr = np.array([1.2, 2.5, 3.8])
print('floor:', np.floor(arr))
print('ceil:', np.ceil(arr))
print('round:', np.round(arr))


`np.sign(arr)` — sign of each element (-1, 0, 1)


In [ ]:
arr = np.array([-5, 0, 5])
print('sign:', np.sign(arr))


## Section 9: Linear Algebra — Matrix Multiply
`dot` vs `matmul` vs `@` operator — differences and when to use each


`np.dot(a, b)` — dot product of 1D vectors


In [ ]:
a = np.array([1, 2])
b = np.array([3, 4])
print('Dot product:', np.dot(a, b))


`np.dot(A, B)` — matrix multiplication of 2D arrays


In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print('Matrix multiplication:\n', np.dot(A, B))


`A @ B` — recommended modern way (same as matmul for 2D)


In [ ]:
print('A @ B:\n', A @ B)


`np.matmul(A, B)` — batched matrix multiply (works for 3D)


In [ ]:
batch_A = np.random.rand(2, 3, 4)
batch_B = np.random.rand(2, 4, 5)
print('Batched matmul shape:', np.matmul(batch_A, batch_B).shape)


Real ML use — forward pass: `output = X @ W + b`


In [ ]:
X = np.random.rand(10, 5) # 10 samples, 5 features
W = np.random.rand(5, 3)  # 5 in, 3 out
b = np.random.rand(3)     # 3 biases
output = X @ W + b
print('Output shape:', output.shape)


## Section 10: Linear Algebra — Other Operations


`np.linalg.det(A)` — determinant


In [ ]:
A = np.array([[1, 2], [3, 4]])
print('Determinant:', np.linalg.det(A))


`np.linalg.inv(A)` — matrix inverse


In [ ]:
print('Inverse:\n', np.linalg.inv(A))


`np.linalg.norm(v)` — L2 norm of a vector


In [ ]:
v = np.array([3, 4])
print('L2 norm:', np.linalg.norm(v))


`np.linalg.norm(v, ord=1)` — L1 norm


In [ ]:
v = np.array([3, -4])
print('L1 norm:', np.linalg.norm(v, ord=1))


`np.linalg.norm(A, 'fro')` — Frobenius norm of a matrix


In [ ]:
A = np.array([[1, 2], [3, 4]])
print('Frobenius norm:', np.linalg.norm(A, 'fro'))


`np.linalg.eig(A)` — eigenvalues and eigenvectors


In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(A)
print('Eigenvalues:', eigenvalues)


`np.linalg.solve(A, b)` — solve system of equations Ax=b


In [ ]:
A = np.array([[3, 1], [1, 2]])
b = np.array([9, 8])
x = np.linalg.solve(A, b)
print('Solution for Ax=b:', x)


`np.trace(A)` — sum of diagonal elements


In [ ]:
A = np.array([[1, 2], [3, 4]])
print('Trace:', np.trace(A))


`np.diag(A)` — extract diagonal OR create diagonal matrix


In [ ]:
A = np.array([[1, 2], [3, 4]])
print('Extract diagonal:', np.diag(A))
print('Create diagonal:\n', np.diag([1, 2, 3]))


`np.linalg.lstsq(A, b, rcond=None)` — least squares solution


In [ ]:
x = np.array([0, 1, 2, 3])
y = np.array([-1, 0.2, 0.9, 2.1])
A = np.vstack([x, np.ones(len(x))]).T
m, c = np.linalg.lstsq(A, y, rcond=None)[0]
print(f'Least squares fit: y = {m:.2f}x + {c:.2f}')


## Section 11: SVD — Singular Value Decomposition
What SVD is — $A = U \Sigma V^T$, used in PCA, recommendation systems, image compression


`np.linalg.svd(A)` — returns U, s, Vt


In [ ]:
A = np.random.rand(4, 5)
U, s, Vt = np.linalg.svd(A)
print('U shape:', U.shape)
print('s shape:', s.shape)
print('Vt shape:', Vt.shape)


Verify A = U @ np.diag(s) @ Vt


In [ ]:
S = np.zeros((4, 5))
S[:4, :4] = np.diag(s)
A_reconstructed = U @ S @ Vt
print('Reconstruction error:', np.linalg.norm(A - A_reconstructed))


Image compression using SVD — keep top K singular values and reconstruct (create a simple matrix to represent an image-like structure)


In [ ]:
img = np.random.rand(50, 50)
img += np.sin(np.linspace(0, 10, 50))[:, None]
U, s, Vt = np.linalg.svd(img)

# Keep top K=5 singular values
k = 5
img_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
print('Reconstructed shape:', img_k.shape)


Plot: original vs reconstructed with different K values


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(img_k, cmap='gray'); axes[1].set_title('K=5')

k = 20
img_20 = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
axes[2].imshow(img_20, cmap='gray'); axes[2].set_title('K=20')
plt.show()


## Section 12: Statistics


`np.mean(arr, axis=0)` — mean of each column (feature mean)


In [ ]:
arr = np.random.rand(10, 3)
print('Column means:', np.mean(arr, axis=0))


`np.std(arr, axis=0)` — std of each column


In [ ]:
print('Column stds:', np.std(arr, axis=0))


`np.var(arr, axis=0)` — variance


In [ ]:
print('Column variances:', np.var(arr, axis=0))


`np.median(arr)` — median


In [ ]:
print('Median:', np.median(arr))


`np.percentile(arr, [25, 50, 75])` — quartiles


In [ ]:
print('Quartiles:', np.percentile(arr, [25, 50, 75]))


`np.quantile(arr, 0.95)` — 95th percentile


In [ ]:
print('95th quantile:', np.quantile(arr, 0.95))


`np.corrcoef(X, Y)` — correlation matrix


In [ ]:
x = np.random.rand(100)
y = 2 * x + np.random.normal(0, 0.1, 100)
print('Correlation matrix:\n', np.corrcoef(x, y))


`np.cov(data)` — covariance matrix


In [ ]:
print('Covariance matrix:\n', np.cov(x, y))


Real ML use — compute feature correlation matrix and plot as heatmap with matplotlib


In [ ]:
X = np.random.rand(100, 4)
corr_matrix = np.corrcoef(X, rowvar=False)
plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar()
plt.title('Feature Correlation Heatmap')
plt.show()


## Section 13: Sorting and Searching


`np.sort(arr)` — returns sorted copy


In [ ]:
arr = np.array([3, 1, 4, 1, 5, 9])
print('Sorted:', np.sort(arr))


`np.sort(arr, axis=0)` — sort each column


In [ ]:
arr = np.array([[3, 2], [1, 4]])
print('Column sorted:\n', np.sort(arr, axis=0))


`np.argsort(arr)` — returns indices that would sort the array


In [ ]:
arr = np.array([3, 1, 4, 1, 5])
indices = np.argsort(arr)
print('Argsort indices:', indices)
print('Sorted using indices:', arr[indices])


Use argsort to get top-K indices (like top predictions)


In [ ]:
probs = np.array([0.1, 0.5, 0.2, 0.9, 0.05])
top_2_indices = np.argsort(probs)[-2:][::-1]
print('Top 2 indices:', top_2_indices)
print('Top 2 probs:', probs[top_2_indices])


`np.unique(arr)` — unique elements


In [ ]:
arr = np.array([1, 2, 2, 3, 1, 4])
print('Unique elements:', np.unique(arr))


`np.unique(arr, return_counts=True)` — unique with counts


In [ ]:
uniques, counts = np.unique(arr, return_counts=True)
print('Uniques:', uniques)
print('Counts:', counts)


`np.bincount(arr)` — count occurrences of each integer


In [ ]:
arr = np.array([0, 1, 1, 3, 2, 1])
print('Bincount:', np.bincount(arr))


`np.searchsorted(sorted_arr, value)` — binary search


In [ ]:
sorted_arr = np.array([1, 3, 5, 7, 9])
index = np.searchsorted(sorted_arr, 4)
print('Insert 4 at index:', index)


## Section 14: Stacking and Splitting


`np.vstack([a, b])` — stack arrays vertically (add rows)


In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print('Vstack:\n', np.vstack([a, b]))


`np.hstack([a, b])` — stack arrays horizontally (add columns)


In [ ]:
print('Hstack:', np.hstack([a, b]))


`np.concatenate([a, b], axis=0)` — general concatenation


In [ ]:
print('Concatenate:\n', np.concatenate([a.reshape(1,3), b.reshape(1,3)], axis=0))


`np.stack([a, b], axis=0)` — new axis stacking


In [ ]:
print('Stack:\n', np.stack([a, b], axis=0))


Real ML use — np.vstack to build a batch from individual samples


In [ ]:
samples = [np.random.rand(5) for _ in range(3)]
batch = np.vstack(samples)
print('Batch shape:', batch.shape)


`np.split(arr, n)` — split into equal parts


In [ ]:
arr = np.arange(9)
print('Split:', np.split(arr, 3))


`np.vsplit(arr, n)` — split rows


In [ ]:
arr = np.arange(16).reshape(4, 4)
print('Vsplit:\n', np.vsplit(arr, 2))


`np.hsplit(arr, n)` — split columns


In [ ]:
print('Hsplit:\n', np.hsplit(arr, 2))


Real ML use — split dataset into train and test


In [ ]:
dataset = np.random.rand(100, 10)
train, test = np.vsplit(dataset, [80])
print('Train shape:', train.shape, 'Test shape:', test.shape)


## Section 15: np.pad()
What padding is — used in CNN convolutions, sequence padding for NLP


`np.pad(arr, pad_width, mode='constant')` — zero padding


In [ ]:
arr = np.array([1, 2, 3])
print('Zero pad:', np.pad(arr, (2, 2), mode='constant'))


`np.pad(arr, ((top, bottom), (left, right)), 'constant')` — asymmetric padding


In [ ]:
arr_2d = np.ones((2, 2))
print('Asymmetric pad:\n', np.pad(arr_2d, ((1, 2), (0, 1)), mode='constant'))


`np.pad` with `mode='reflect'` — reflect padding


In [ ]:
arr = np.array([1, 2, 3, 4])
print('Reflect pad:', np.pad(arr, (2, 2), mode='reflect'))


Real CNN use — pad a 2D image (28x28) → (32x32) with zeros


In [ ]:
image = np.ones((28, 28))
padded_image = np.pad(image, ((2, 2), (2, 2)), mode='constant')
print('Padded image shape:', padded_image.shape)


## Section 16: np.einsum() — Einstein Summation
What einsum is — compact notation for matrix ops used in attention mechanisms, deep learning

Explain the notation: `ij,jk->ik` means matrix multiply $A_{i,j} \times B_{j,k} \rightarrow C_{i,k}$


Matrix multiply with einsum — `np.einsum('ij,jk->ik', A, B)`


In [ ]:
A = np.random.rand(2, 3)
B = np.random.rand(3, 4)
C = np.einsum('ij,jk->ik', A, B)
print('Matrix multiply shape:', C.shape)


Batch matrix multiply — `np.einsum('bij,bjk->bik', A, B)`


In [ ]:
batch_A = np.random.rand(10, 2, 3)
batch_B = np.random.rand(10, 3, 4)
batch_C = np.einsum('bij,bjk->bik', batch_A, batch_B)
print('Batch matrix multiply shape:', batch_C.shape)


Dot product — `np.einsum('i,i->', a, b)`


In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print('Dot product:', np.einsum('i,i->', a, b))


Outer product — `np.einsum('i,j->ij', a, b)`


In [ ]:
print('Outer product:\n', np.einsum('i,j->ij', a, b))


Trace — `np.einsum('ii->', A)`


In [ ]:
A = np.arange(9).reshape(3, 3)
print('Trace:', np.einsum('ii->', A))


Transpose — `np.einsum('ij->ji', A)`


In [ ]:
print('Transpose:\n', np.einsum('ij->ji', A))


Real ML use — attention scores: `np.einsum('bd,nd->bn', query, keys)`


In [ ]:
query = np.random.rand(8, 64) # Batch, d_model
keys = np.random.rand(10, 64) # N_seq, d_model
scores = np.einsum('bd,nd->bn', query, keys)
print('Attention scores shape:', scores.shape)


## Section 17: np.vectorize()


Define a regular Python function that works on scalars


In [ ]:
def my_func(x):
    if x > 0:
        return x ** 2
    else:
        return 0


`np.vectorize(func)` — make it work on arrays


In [ ]:
vec_func = np.vectorize(my_func)
arr = np.array([-2, -1, 0, 1, 2])
print('Vectorized output:', vec_func(arr))


Compare speed: Python loop vs np.vectorize vs true numpy


In [ ]:
import time
large_arr = np.random.randn(100000)

t0 = time.time()
_ = [my_func(x) for x in large_arr]
t1 = time.time()

t2 = time.time()
_ = vec_func(large_arr)
t3 = time.time()

t4 = time.time()
_ = np.where(large_arr > 0, large_arr ** 2, 0)
t5 = time.time()

print(f'List comp: {t1-t0:.4f}s')
print(f'np.vectorize: {t3-t2:.4f}s')
print(f'True NumPy: {t5-t4:.4f}s')


Real use — apply custom preprocessing function to entire array


In [ ]:
import re
def clean_text(s):
    return re.sub(r'[^a-zA-Z]', '', str(s)).lower()

vec_clean = np.vectorize(clean_text)
texts = np.array(['Hello 123!', 'NumPy is GR8.', 'Test...'])
print('Cleaned text:', vec_clean(texts))


## Section 18: Random Module (Deep Dive)


`np.random.seed(42)` — reproducibility


In [ ]:
np.random.seed(42)
print('Random:', np.random.rand())


`np.random.shuffle(arr)` — in-place shuffle


In [ ]:
arr = np.arange(10)
np.random.shuffle(arr)
print('Shuffled:', arr)


`np.random.permutation(n)` — random permutation (returns new array)


In [ ]:
print('Permutation:', np.random.permutation(10))


`np.random.choice(arr, size, replace)` — random sampling


In [ ]:
arr = np.array(['A', 'B', 'C', 'D'])
print('Choice:', np.random.choice(arr, size=2, replace=False))


`np.random.normal(mu, sigma, size)`


In [ ]:
norm_data = np.random.normal(0, 1, 1000)


`np.random.uniform(low, high, size)`


In [ ]:
uni_data = np.random.uniform(0, 10, 1000)


`np.random.binomial(n, p, size)`


In [ ]:
binom_data = np.random.binomial(10, 0.5, 1000)


`np.random.poisson(lam, size)`


In [ ]:
poisson_data = np.random.poisson(3, 1000)


`np.random.exponential(scale, size)`


In [ ]:
exp_data = np.random.exponential(1.0, 1000)


Plot all distributions as histograms in a grid (matplotlib)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].hist(norm_data, bins=30); axes[0, 0].set_title('Normal')
axes[0, 1].hist(uni_data, bins=30); axes[0, 1].set_title('Uniform')
axes[0, 2].hist(binom_data, bins=11); axes[0, 2].set_title('Binomial')
axes[1, 0].hist(poisson_data, bins=15); axes[1, 0].set_title('Poisson')
axes[1, 1].hist(exp_data, bins=30); axes[1, 1].set_title('Exponential')
axes[1, 2].axis('off')
plt.tight_layout()
plt.show()


## Section 19: Copy vs View
The most common NumPy trap! Slices are VIEWS not copies!


Demonstrate slice is a view — modifying slice modifies original


In [ ]:
arr = np.arange(5)
slice_arr = arr[1:4]
slice_arr[0] = 99
print('Original modified:', arr)


`.copy()` — create a true copy


In [ ]:
arr = np.arange(5)
copy_arr = arr[1:4].copy()
copy_arr[0] = 99
print('Original intact:', arr)


When operations return views vs copies


In [ ]:
# Slicing returns views. Fancy indexing returns copies.
arr = np.arange(5)
fancy = arr[[1, 2, 3]]
fancy[0] = 99
print('Original intact (fancy index):', arr)


`np.shares_memory(a, b)` — check if two arrays share memory


In [ ]:
print('Shares memory (slice):', np.shares_memory(arr, arr[1:4]))
print('Shares memory (copy):', np.shares_memory(arr, arr.copy()))


Benchmark — view vs copy memory usage


In [ ]:
large_arr = np.zeros(10000000)
view = large_arr[:]
copy_val = large_arr.copy()
print('View shares memory:', np.shares_memory(large_arr, view))
print('Copy shares memory:', np.shares_memory(large_arr, copy_val))


Real ML danger — accidentally modifying training data through a view


In [ ]:
X_train = np.ones((10, 5))
batch = X_train[:3]
batch *= 2  # modifies X_train!
print('X_train top row:\n', X_train[0])


## Section 20: Cheat Sheet
- `arr[mask]` : boolean indexing
- `arr[[idx1, idx2]]` : fancy indexing
- `np.where(cond, x, y)` : if-else for arrays
- `arr.clip(min, max)` : clamp values
- `A @ B` : matrix multiplication
- `np.linalg.svd(A)` : singular value decomposition
- `np.einsum()` : Einstein summation
- `arr.copy()` : true copy


## Section 21: Exercises


Exercise 1: Boolean indexing on a dataset
Given an array `ages`, extract all ages between 18 and 30 inclusive.


In [ ]:
ages = np.array([12, 18, 25, 30, 45, 17, 21])
# Try it yourself


In [ ]:
ages = np.array([12, 18, 25, 30, 45, 17, 21])
# Solution
res = ages[(ages >= 18) & (ages <= 30)]
print('Ex 1:', res)


Exercise 2: Broadcasting — normalize a matrix
Given a 2D matrix, subtract the minimum of each column and divide by the range (max - min) of each column.


In [ ]:
matrix = np.random.rand(5, 3) * 10
# Try it yourself


In [ ]:
matrix = np.random.rand(5, 3) * 10
# Solution
col_min = matrix.min(axis=0)
col_max = matrix.max(axis=0)
normalized = (matrix - col_min) / (col_max - col_min)
print('Ex 2:\n', normalized)


Exercise 3: Linear algebra — solve a system of equations
Solve 2x + 3y = 8 and 3x - y = 1


In [ ]:
# Try it yourself


In [ ]:
# Solution
A = np.array([[2, 3], [3, -1]])
b = np.array([8, 1])
x = np.linalg.solve(A, b)
print('Ex 3:', x)


Exercise 4: Statistics — compute correlation matrix
Compute the correlation between array x and array y without using `np.corrcoef`.


In [ ]:
x = np.random.randn(100)
y = x * 2 + np.random.randn(100)
# Try it yourself


In [ ]:
x = np.random.randn(100)
y = x * 2 + np.random.randn(100)
# Solution
x_centered = x - x.mean()
y_centered = y - y.mean()
cov = np.mean(x_centered * y_centered)
corr = cov / (x.std() * y.std())
print('Ex 4 (manual corr):', corr)
print('Check with np.corrcoef:', np.corrcoef(x, y)[0, 1])


Exercise 5: einsum — implement matrix multiply and batch multiply
Use einsum to compute the batched matrix multiplication of `A` (shape B, M, K) and `B` (shape B, K, N).


In [ ]:
A = np.random.rand(10, 5, 4)
B = np.random.rand(10, 4, 3)
# Try it yourself


In [ ]:
A = np.random.rand(10, 5, 4)
B = np.random.rand(10, 4, 3)
# Solution
C = np.einsum('bmk,bkn->bmn', A, B)
print('Ex 5 shape:', C.shape)
